In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import DenseNet121

In [3]:
import os
data = pd.read_csv("../Dataset/HAM10000_metadata.csv")

In [4]:
image_dir1 = "../Dataset/HAM10000_images_part_1"
image_dir2 = "../Dataset/HAM10000_images_part_2"

image_paths = {}

for folder in [image_dir1, image_dir2]:

    for img in os.listdir(folder):

        image_id = img.split(".")[0]

        image_paths[image_id] = os.path.join(folder,img)

data["path"] = data["image_id"].map(image_paths)

data.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,../Dataset/HAM10000_images_part_1/ISIC_0027419...
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,../Dataset/HAM10000_images_part_1/ISIC_0025030...
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,../Dataset/HAM10000_images_part_1/ISIC_0026769...
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,../Dataset/HAM10000_images_part_1/ISIC_0025661...
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,../Dataset/HAM10000_images_part_2/ISIC_0031633...


In [5]:
train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42, stratify=data["dx"] )

In [6]:
train_df

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
4357,HAM_0000946,ISIC_0031775,nv,follow_up,60.0,male,trunk,../Dataset/HAM10000_images_part_2/ISIC_0031775...
1751,HAM_0006097,ISIC_0027306,mel,histo,60.0,male,chest,../Dataset/HAM10000_images_part_1/ISIC_0027306...
9527,HAM_0004348,ISIC_0033895,nv,consensus,40.0,female,unknown,../Dataset/HAM10000_images_part_2/ISIC_0033895...
8311,HAM_0006608,ISIC_0025491,nv,histo,60.0,male,back,../Dataset/HAM10000_images_part_1/ISIC_0025491...
1214,HAM_0005678,ISIC_0031023,mel,histo,60.0,male,chest,../Dataset/HAM10000_images_part_2/ISIC_0031023...
...,...,...,...,...,...,...,...,...
492,HAM_0001605,ISIC_0024422,bkl,histo,75.0,male,upper extremity,../Dataset/HAM10000_images_part_1/ISIC_0024422...
7092,HAM_0005143,ISIC_0030303,nv,histo,40.0,male,chest,../Dataset/HAM10000_images_part_2/ISIC_0030303...
9254,HAM_0002364,ISIC_0029838,nv,consensus,5.0,female,hand,../Dataset/HAM10000_images_part_2/ISIC_0029838...
5674,HAM_0005583,ISIC_0025574,nv,follow_up,50.0,male,abdomen,../Dataset/HAM10000_images_part_1/ISIC_0025574...


In [7]:
valid_df, test_df = train_test_split(temp_df , test_size= 0.5, random_state= 42, stratify=temp_df["dx"])

In [8]:
valid_df

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
1800,HAM_0002841,ISIC_0032982,mel,histo,20.0,male,back,../Dataset/HAM10000_images_part_2/ISIC_0032982...
4180,HAM_0000885,ISIC_0025293,nv,follow_up,40.0,female,upper extremity,../Dataset/HAM10000_images_part_1/ISIC_0025293...
3249,HAM_0004544,ISIC_0031099,nv,follow_up,50.0,male,foot,../Dataset/HAM10000_images_part_2/ISIC_0031099...
624,HAM_0002548,ISIC_0028656,bkl,histo,50.0,female,lower extremity,../Dataset/HAM10000_images_part_1/ISIC_0028656...
319,HAM_0007260,ISIC_0028336,bkl,histo,60.0,male,chest,../Dataset/HAM10000_images_part_1/ISIC_0028336...
...,...,...,...,...,...,...,...,...
6954,HAM_0000324,ISIC_0029945,nv,histo,80.0,male,back,../Dataset/HAM10000_images_part_2/ISIC_0029945...
1473,HAM_0005490,ISIC_0033931,mel,histo,70.0,male,upper extremity,../Dataset/HAM10000_images_part_2/ISIC_0033931...
4717,HAM_0001514,ISIC_0024659,nv,follow_up,60.0,male,trunk,../Dataset/HAM10000_images_part_1/ISIC_0024659...
4366,HAM_0002654,ISIC_0025698,nv,follow_up,40.0,female,trunk,../Dataset/HAM10000_images_part_1/ISIC_0025698...


In [9]:
test_df

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
547,HAM_0005335,ISIC_0031580,bkl,histo,40.0,male,upper extremity,../Dataset/HAM10000_images_part_2/ISIC_0031580...
8877,HAM_0006835,ISIC_0031594,nv,histo,75.0,male,chest,../Dataset/HAM10000_images_part_2/ISIC_0031594...
3696,HAM_0007205,ISIC_0026269,nv,follow_up,60.0,male,lower extremity,../Dataset/HAM10000_images_part_1/ISIC_0026269...
5850,HAM_0004072,ISIC_0032130,nv,follow_up,60.0,male,upper extremity,../Dataset/HAM10000_images_part_2/ISIC_0032130...
3371,HAM_0003341,ISIC_0027212,nv,follow_up,55.0,male,lower extremity,../Dataset/HAM10000_images_part_1/ISIC_0027212...
...,...,...,...,...,...,...,...,...
3333,HAM_0000642,ISIC_0026520,nv,follow_up,45.0,female,lower extremity,../Dataset/HAM10000_images_part_1/ISIC_0026520...
7734,HAM_0006567,ISIC_0032649,nv,histo,30.0,female,back,../Dataset/HAM10000_images_part_2/ISIC_0032649...
3636,HAM_0003924,ISIC_0030569,nv,follow_up,80.0,male,trunk,../Dataset/HAM10000_images_part_2/ISIC_0030569...
4899,HAM_0001089,ISIC_0029941,nv,follow_up,40.0,male,trunk,../Dataset/HAM10000_images_part_2/ISIC_0029941...


# Encode

In [10]:
label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

In [11]:
train_df["label"] 

4357    5
1751    4
9527    5
8311    5
1214    4
       ..
492     2
7092    5
9254    5
5674    5
3186    5
Name: label, Length: 7010, dtype: int64

In [12]:
NUM_CLASSES

7

## Compute Class Weights

In [13]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


## Create Image Loader

In [14]:
IMG_SIZE = (224, 224)

def process_image(path, label):

    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)

    image = tf.image.resize(image, IMG_SIZE)

    return image, label

# Data Augumentation

In [ ]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.2),

    tf.keras.layers.RandomZoom(0.2),

    tf.keras.layers.RandomContrast(0.2)

])

E0000 00:00:1785311230.054110  489683 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [17]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["path"], train_df["label"])
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.batch(32)
train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

In [19]:
# Validation

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (valid_df["path"], valid_df["label"])
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)


In [ ]:
test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_df["path"], test_df["label"])
)

test_dataset = test_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [21]:
BATCH_SIZE = 32

valid_dataset = valid_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.batch(BATCH_SIZE)

valid_dataset = valid_dataset.prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

In [22]:
valid_dataset

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

# Basic CNN

In [25]:
basic_cnn = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(224,224,3)),

    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(
            128,
            (3,3),
            activation="relu"
        ),
    
    tf.keras.layers.MaxPooling2D((2,2)),
    

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        256,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

W0000 00:00:1785313615.717674  489683 cpu_allocator_impl.cc:82] Allocation of 88604672 exceeds 10% of free system memory.
W0000 00:00:1785313615.906408  489683 cpu_allocator_impl.cc:82] Allocation of 88604672 exceeds 10% of free system memory.


In [26]:
basic_cnn.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
basic_cnn.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

history_basic = basic_cnn.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=5,

    class_weight=class_weights,

)

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/Skin_Disease_Classification___Diagnosis_Pl-WW3Rx1qq/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 791s 4s/step - accuracy: 0.1124 - loss: 26.7603 - val_accuracy: 0.0659 - val_loss: 1.9622
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 742s 3s/step - accuracy: 0.0653 - loss: 2.0659 - val_accuracy: 0.0313 - val_loss: 1.9725
Epoch 3/5
100/220 ━━━━━━━━━━━━━━━━━━━━ 5:45 3s/step - accuracy: 0.0584 - loss: 1.9761

KeyboardInterrupt: 